<a href="https://colab.research.google.com/github/dennisgathu8/36CHAMBERS/blob/main/Prdictive_Keyboard_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install nltk

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from google.colab import files

nltk.download('punkt')
nltk.download('punkt_tab')

uploaded = files.upload()
file_name = list(uploaded.keys())[0]
text = uploaded[file_name].decode('utf-8').lower()

tokens = word_tokenize(text)
print("Total Tokens:", len(tokens))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Saving sherlock-holm.es_stories_plain-text_advs.txt to sherlock-holm.es_stories_plain-text_advs (1).txt
Total Tokens: 125772


In [ ]:
# creating a vocabulary
from collections import Counter
word_counts = Counter(tokens)
vocab = sorted(word_counts, key=word_counts.get, reverse=True)

word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}
vocab_size = len(vocab)

In [ ]:
!pip install torch
import torch
# building the output sequences
sequence_length = 4 # eg. "I am going to [predict this]"
data = []
for i in range(len(tokens) - sequence_length):
  input_seq = tokens[i:i + sequence_length - 1]
  target = tokens[i + sequence_length - 1]
  data.append((input_seq, target))

# convert words to indices
def encode(seq): return [word2idx[word] for word in seq]

encoded_data = [(torch.tensor(encode(inp)), torch.tensor(word2idx[target])) for inp, target in data]

In [ ]:
# designing the model architecture
import torch.nn as nn
class PredictiveKeyboard(nn.Module):
  def __init__(self, vocab_size, embed_dim=64, hidden_dim=128):
    super(PredictiveKeyboard, self).__init__()
    self.embedding = nn.Embedding(vocab_size, embed_dim)
    self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
    self.fc = nn.Linear(hidden_dim, vocab_size)

  def forward(self, x):
    x = self.embedding(x)
    output, _ = self.lstm(x)
    output = self.fc(output[:, -1, :]) #last LSTM output
    return output

In [ ]:
import torch
import torch.optim as optim
import random

model = PredictiveKeyboard(vocab_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

epochs = 20
for epoch in range(epochs):
    total_loss = 0
    random.shuffle(encoded_data)
    for input_seq, target in encoded_data[:10000]:  # Limit data for speed
        input_seq = input_seq.unsqueeze(0)
        output = model(input_seq)
        loss = criterion(output, target.unsqueeze(0))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 66551.2095
Epoch 2, Loss: 66237.7339
Epoch 3, Loss: 69264.5142
Epoch 4, Loss: 70745.8935
Epoch 5, Loss: 72287.1165
Epoch 6, Loss: 72453.8722
Epoch 7, Loss: 73678.1628
Epoch 8, Loss: 73749.9455
Epoch 9, Loss: 74814.1760
Epoch 10, Loss: 75485.9297
Epoch 11, Loss: 78199.8554
Epoch 12, Loss: 78757.5330
Epoch 13, Loss: 78023.9261
Epoch 14, Loss: 77772.0783
Epoch 15, Loss: 80090.7739
Epoch 16, Loss: 79655.3075
Epoch 17, Loss: 79104.8189
Epoch 18, Loss: 82044.0499
Epoch 19, Loss: 80668.0440
Epoch 20, Loss: 83857.5927


In [ ]:
import torch.nn.functional as F
def suggest_next_words(model, text_prompt, top_k=3):
  model.eval()
  tokens = word_tokenize(text_prompt.lower())
  if len(tokens) < sequence_length - 1:
    raise ValueError(f"Input should be atleast {sequence_length - 1} words long.")

  input_seq = tokens[-(sequence_length - 1):]
  input_tensor = torch.tensor(encode(input_seq)).unsqueeze(0)

  with torch.no_grad():
    output = model(input_tensor)
    probs = F.softmax(output, dim=1).squeeze()
    top_indices = torch.topk(probs, top_k).indices.tolist()

  return [idx2word[idx] for idx in top_indices]

print("Suggestions:", suggest_next_words(model, "So, are we really at"))

Suggestions: ['me', 'all', 'once']
